In [13]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/media/ehsan/MyFiles2/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting"
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [14]:
from config import load_config

cfg = load_config(PROJECT_ROOT / "configs/config.yaml")

cfg["data"]["root"] = (
    "/media/ehsan/MyFiles2/FraUAS/HIS Project/PTB-XL"
)

print(cfg["data"])

{'root': '/media/ehsan/MyFiles2/FraUAS/HIS Project/PTB-XL', 'sampling_rate': 100, 'history': 700, 'horizon': 300, 'synthetic': False, 'start': 0, 'layout': 'time_first'}


In [15]:
cfg["run_mode"] = "medium"

In [16]:
from data import prepare

cache, manifest = prepare(cfg)

print("Cache:", cache)
print("Splits:", manifest["splits"])

Cache: data/study_v3/ce45446649a4387a
Splits: {'train': {'records': 1731, 'patients': 1500, 'windows': 1731}, 'val': {'records': 2183, 'patients': 1942, 'windows': 2183}}


In [5]:
from data import ECGDataset
from preprocessing import feature_arrays

arrays, names, details = feature_arrays(
    cfg,
    cache,
    "F0",
    splits=("train", "val"),
)

dataset = ECGDataset(
    cache,
    cfg,
    "train",
    "P0",
    arrays["train"],
)

sample = dataset[0]

print("x:", sample["x"].shape)
print("y:", sample["y"].shape)
print("static:", sample["static"].shape)

x: torch.Size([700, 12])
y: torch.Size([300, 12])
static: torch.Size([0])


In [6]:
from models import build_model
import torch

model = build_model(
    "tft",
    static_dim=0,
    horizon=cfg["data"]["horizon"],
    **cfg["model"]
)

print(model)

TFTForecaster(
  (input_projection): Linear(in_features=12, out_features=64, bias=True)
  (grn_stack): ModuleList(
    (0): GatedResidualNetwork(
      (fc1): Linear(in_features=64, out_features=64, bias=True)
      (fc2): Linear(in_features=64, out_features=64, bias=True)
      (dropout): Dropout(p=0.2, inplace=False)
      (gate): Sequential(
        (0): Linear(in_features=64, out_features=64, bias=True)
        (1): Sigmoid()
      )
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (attention): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,

In [7]:
x = torch.randn(2, 700, 12)

y = model(x)

print(y.shape)

torch.Size([2, 300, 12])


In [8]:
from train import train_run

run = train_run(
    cfg,
    cache,
    architecture="tft",
    preprocessing="P0",
    feature_set="F0",
    epochs=1,
    name="tft_smoke"
)

print(run)

tft_smoke: epoch 1/1; validation patient MAE 0.115988 mV
results/primary_7s3s_v3/pilot/checkpoints/tft_smoke


In [9]:
run = train_run(
    cfg,
    cache,
    architecture="tft",
    preprocessing="P1",
    feature_set="F0",
    epochs=1,
    name="tft_p1_smoke"
)


tft_p1_smoke: epoch 1/1; validation patient MAE 0.115876 mV


In [10]:
cfg["run_mode"] = "medium"

cache, manifest = prepare(cfg)

run_tft_p0 = train_run(
    cfg,
    cache,
    architecture="tft",
    preprocessing="P0",
    feature_set="F0",
    epochs=120,
    name="tft_P0_F0_full_120e"
)


tft_P0_F0_full_120e: epoch 1/120; validation patient MAE 0.110971 mV
tft_P0_F0_full_120e: epoch 2/120; validation patient MAE 0.109127 mV
tft_P0_F0_full_120e: epoch 3/120; validation patient MAE 0.107699 mV
tft_P0_F0_full_120e: epoch 4/120; validation patient MAE 0.105478 mV
tft_P0_F0_full_120e: epoch 5/120; validation patient MAE 0.104998 mV
tft_P0_F0_full_120e: epoch 6/120; validation patient MAE 0.105887 mV
tft_P0_F0_full_120e: epoch 7/120; validation patient MAE 0.105489 mV
tft_P0_F0_full_120e: epoch 8/120; validation patient MAE 0.103944 mV
tft_P0_F0_full_120e: epoch 9/120; validation patient MAE 0.105435 mV
tft_P0_F0_full_120e: epoch 10/120; validation patient MAE 0.105100 mV
tft_P0_F0_full_120e: epoch 11/120; validation patient MAE 0.105089 mV
tft_P0_F0_full_120e: epoch 12/120; validation patient MAE 0.105172 mV
tft_P0_F0_full_120e: epoch 13/120; validation patient MAE 0.105476 mV
tft_P0_F0_full_120e: epoch 14/120; validation patient MAE 0.104576 mV


KeyboardInterrupt: 

In [17]:
#TFT Medium P0 60 Epochs
 
from train import train_run

run_tft_p0 = train_run(
    cfg,
    cache,
    architecture="tft",
    preprocessing="P0",
    feature_set="F0",
    epochs=60,
    name="tft_P0_F0_medium_60e"
)

print(run_tft_p0)

tft_P0_F0_medium_60e: epoch 1/60; validation patient MAE 0.114207 mV
tft_P0_F0_medium_60e: epoch 2/60; validation patient MAE 0.114067 mV
tft_P0_F0_medium_60e: epoch 3/60; validation patient MAE 0.114008 mV
tft_P0_F0_medium_60e: epoch 4/60; validation patient MAE 0.113895 mV
tft_P0_F0_medium_60e: epoch 5/60; validation patient MAE 0.113588 mV
tft_P0_F0_medium_60e: epoch 6/60; validation patient MAE 0.113053 mV
tft_P0_F0_medium_60e: epoch 7/60; validation patient MAE 0.113084 mV
tft_P0_F0_medium_60e: epoch 8/60; validation patient MAE 0.112614 mV
tft_P0_F0_medium_60e: epoch 9/60; validation patient MAE 0.112609 mV
tft_P0_F0_medium_60e: epoch 10/60; validation patient MAE 0.112324 mV
tft_P0_F0_medium_60e: epoch 11/60; validation patient MAE 0.111902 mV
tft_P0_F0_medium_60e: epoch 12/60; validation patient MAE 0.111988 mV
tft_P0_F0_medium_60e: epoch 13/60; validation patient MAE 0.110966 mV
tft_P0_F0_medium_60e: epoch 14/60; validation patient MAE 0.110909 mV
tft_P0_F0_medium_60e: epoch 1

In [19]:
#TFT Medium P1 60 Epochs

from train import train_run

run_tft_p1 = train_run(
    cfg,
    cache,
    architecture="tft",
    preprocessing="P1",
    feature_set="F0",
    epochs=60,
    name="tft_P1_F0_medium_60e"
)

print(run_tft_p1)

tft_P1_F0_medium_60e: epoch 1/60; validation patient MAE 0.114218 mV
tft_P1_F0_medium_60e: epoch 2/60; validation patient MAE 0.114136 mV
tft_P1_F0_medium_60e: epoch 3/60; validation patient MAE 0.114070 mV
tft_P1_F0_medium_60e: epoch 4/60; validation patient MAE 0.114026 mV
tft_P1_F0_medium_60e: epoch 5/60; validation patient MAE 0.114019 mV
tft_P1_F0_medium_60e: epoch 6/60; validation patient MAE 0.114042 mV
tft_P1_F0_medium_60e: epoch 7/60; validation patient MAE 0.114069 mV
tft_P1_F0_medium_60e: epoch 8/60; validation patient MAE 0.114001 mV
tft_P1_F0_medium_60e: epoch 9/60; validation patient MAE 0.113970 mV
tft_P1_F0_medium_60e: epoch 10/60; validation patient MAE 0.113966 mV
tft_P1_F0_medium_60e: epoch 11/60; validation patient MAE 0.114008 mV
tft_P1_F0_medium_60e: epoch 12/60; validation patient MAE 0.113963 mV
tft_P1_F0_medium_60e: epoch 13/60; validation patient MAE 0.114572 mV
tft_P1_F0_medium_60e: epoch 14/60; validation patient MAE 0.114431 mV
tft_P1_F0_medium_60e: epoch 1